# CSE4078 Spring 2026 — Group 5 — Deadline 2
## QLoRA Fine-Tuning of Qwen3-4B for Turkish Medical QA

**Members:** Yiğitcan Değişme, Emir Devran, Kerem Kolay, Yunus Emre Gül, Ömer Faruk Sevinç  
**Dataset:** alibayram/doktorsitesi  
**Model:** Qwen/Qwen3-4B  
**Platform:** Google Colab Pro — NVIDIA A100 40GB GPU

---
### Notebook Structure
1. **Data Preprocessing** — Filter, deduplicate, stratified split, save JSONL
2. **Model Setup** — Load Qwen3-4B with QLoRA (4-bit NF4)
3. **SFT Data Preparation** — 15,000-example stratified subsample + Alpaca format
4. **Fine-Tuning** — SFTTrainer, 1 epoch
5. **Save Model** — Save LoRA adapter
6. **Post-SFT Inference** — Run fine-tuned model on benchmark_test.jsonl
7. **Evaluation** — Compute all 8 metrics (Exact Match, Token F1, ROUGE-1/2/L, MiniLM, BERTScore, Length)

---
## 1. Data Preprocessing

In [ ]:
!pip install datasets pandas scikit-learn huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

print("Veri kümesi Hugging Face'ten güvenli şekilde indiriliyor, lütfen bekleyin...")

# 1. Kilitli veri kümesini token kullanarak çekiyoruz
dataset = load_dataset("alibayram/doktorsitesi")
df = pd.DataFrame(dataset['train'])

# 2. Ödev dokümanında belirtilen zorunlu 9 branş
hedef_branslar = [
    "beyin-ve-sinir-cerrahisi",
    "uroloji",
    "ortopedi-ve-travmatoloji",
    "dahiliye-ve-ic-hastaliklari",
    "genel-cerrahi",
    "kulak-burun-bogaz-hastaliklari",
    "fiziksel-tip-ve-rehabilitasyon",
    "kardiyoloji",
    "kalp-damar-cerrahisi"
]

# Sadece bu branşları filtreliyoruz
df_filtered = df[df['doctor_speciality'].isin(hedef_branslar)].copy()

# 3. Deduplikasyon
before = len(df_filtered)
df_filtered = df_filtered.drop_duplicates(subset=['question_content'])
print(f"Silinen tekrar: {before - len(df_filtered)}")

# 4. Stratified split (seed=42 — reproducible)
train_df, test_df = train_test_split(
    df_filtered,
    test_size=0.1,
    stratify=df_filtered['doctor_speciality'],
    random_state=42
)

print("\n" + "="*40)
print(f"Filtrelenmiş Toplam Veri Sayısı : {len(df_filtered)}")
print(f"Eğitim Seti (sft_train)         : {len(train_df)}")
print(f"Test Seti   (benchmark_test)    : {len(test_df)}")
print("="*40)

# 5. JSONL formatında kaydet
import json

instruction = "Aşağıdaki tıbbi soruyu Türkçe ve dikkatli biçimde yanıtla."

with open("sft_train.jsonl", "w", encoding="utf-8") as f:
    for _, row in train_df.iterrows():
        record = {
            "instruction": instruction,
            "input": row["question_content"],
            "output": row["question_answer"],
            "doctor_speciality": row["doctor_speciality"]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

with open("benchmark_test.jsonl", "w", encoding="utf-8") as f:
    for _, row in test_df.iterrows():
        record = {
            "instruction": instruction,
            "input": row["question_content"],
            "output": row["question_answer"],
            "doctor_speciality": row["doctor_speciality"]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("sft_train.jsonl ve benchmark_test.jsonl kaydedildi!")

# Train/test overlap kontrolü
train_q = set(train_df['question_content'].tolist())
test_q  = set(test_df['question_content'].tolist())
print(f"Train-Test overlap: {len(train_q & test_q)} (0 olmalı)")

---
## 2. Model Setup — Qwen3-4B with QLoRA

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerator transformers

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Maksimum token sayısı
dtype = None           # Otomatik saptama (bf16/fp16)
load_in_4bit = True    # 4-bit NF4 quantization (QLoRA)

# Qwen3-4B base modelini yükle
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-4B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# LoRA adaptörlerini ekle
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # LoRA rank
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("Qwen3-4B modeli QLoRA eğitimi için hazırlandı!")

---
## 3. SFT Data Preparation — 15,000 Stratified Subsample

In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Tam train havuzunu oku
tum_train_df = pd.read_csv("sft_train.csv")

# Ödev kuralı: branş dağılımını koruyarak tam 15.000 örnek seç
oranti = 15000 / len(tum_train_df)
train_15k_df, _ = train_test_split(
    tum_train_df,
    train_size=oranti,
    stratify=tum_train_df['doctor_speciality'],
    random_state=42
)

print(f"Tam train havuzu : {len(tum_train_df)}")
print(f"SFT subsample    : {len(train_15k_df)} (ödev kuralı: min 15,000)")
print(f"\nBranş dağılımı:")
print(train_15k_df['doctor_speciality'].value_counts())

# Alpaca formatına çevir
alpaca_prompt = """Aşağıda bir hastanın tıbbi sorusu ve bir uzman doktorun buna verdiği yanıt yer almaktadır.

### Soru:
{}

### Cevap:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    questions = examples["question_content"]
    answers   = examples["question_answer"]
    texts = []
    for q, a in zip(questions, answers):
        text = alpaca_prompt.format(q, a) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_pandas(train_15k_df)
dataset = dataset.map(formatting_prompts_func, batched=True)
print("\nVeri kümesi SFT formatına dönüştürüldü!")

---
## 4. Fine-Tuning — SFTTrainer, 1 Epoch

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,   # Effective batch = 8
        warmup_steps = 5,
        num_train_epochs = 1,
        save_strategy = "no",
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

torch.cuda.empty_cache()
print("Eğitim başlatılıyor — 15,000 örnek, 1 epoch...")
trainer_stats = trainer.train()
print("Eğitim tamamlandı!")

---
## 5. Save Fine-Tuned Model

In [ ]:
model.save_pretrained("qwen3_doc_lora_15k")
tokenizer.save_pretrained("qwen3_doc_lora_15k")
print("Model 'qwen3_doc_lora_15k' olarak kaydedildi!")

# Google Drive'a yedekle (opsiyonel)
# import shutil
# shutil.copytree('qwen3_doc_lora_15k', '/content/drive/MyDrive/qwen3_doc_lora_15k')

---
## 6. Post-SFT Inference on benchmark_test.jsonl

In [ ]:
from unsloth import FastLanguageModel
import pandas as pd
import torch
import json
import os

# Fine-tune edilmiş modeli yükle
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen3_doc_lora_15k",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# Benchmark test setini yükle (1,500 satır — eğitimde hiç kullanılmadı)
records = [json.loads(l) for l in open("benchmark_test.jsonl", encoding="utf-8")]
print(f"Test seti: {len(records)} satır")

def generate_answer(question):
    """Deadline 1 ile birebir aynı inference ayarları."""
    messages = [
        {"role": "system", "content": "Asagidaki tibbi soruyu Turkce ve dikkatli bicimde yanitla."},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False   # Qwen3 thinking modunu kapat
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=96,       # Deadline 1 ile aynı
            do_sample=False,         # Greedy decoding — Deadline 1 ile aynı
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Checkpoint ile inference (kesilirse kaldığı yerden devam eder)
out_file = "qwen3_finetuned_predictions.csv"

if os.path.exists(out_file):
    done_df = pd.read_csv(out_file)
    done_indices = set(done_df["index"].tolist())
    rows = done_df.to_dict("records")
    print(f"Kaldığı yerden devam: {len(done_indices)} tamamlanmış")
else:
    done_indices = set()
    rows = []

for i, rec in enumerate(records):
    if i in done_indices:
        continue
    pred = generate_answer(rec["input"])
    rows.append({
        "index": i,
        "doctor_speciality": rec["doctor_speciality"],
        "question": rec["input"],
        "reference": rec["output"],
        "prediction": pred,
    })
    if i % 50 == 0:
        pd.DataFrame(rows).to_csv(out_file, index=False)
        print(f"{i}/1500 tamamlandı...")

pd.DataFrame(rows).to_csv(out_file, index=False)
print("Inference tamamlandı! -> qwen3_finetuned_predictions.csv")

---
## 7. Evaluation — All 8 Metrics

In [ ]:
!pip install rouge-score bert-score sentence-transformers -q

from rouge_score import rouge_scorer
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np
import unicodedata, re

df = pd.read_csv("qwen3_finetuned_predictions.csv")
print(f"Toplam satır: {len(df)}")

references  = df['reference'].fillna("").tolist()
predictions = df['prediction'].fillna("").tolist()

# --- Normalizasyon (Deadline 1 ile aynı: NFKD + casefold + punct removal) ---
def normalize(text):
    text = unicodedata.normalize("NFKD", str(text))
    text = text.casefold()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# --- Exact Match ---
df['exact_match'] = [int(normalize(r) == normalize(p))
                     for r, p in zip(references, predictions)]

# --- Token F1 ---
def token_f1(ref, pred):
    r = normalize(ref).split()
    p = normalize(pred).split()
    if not r or not p:
        return 0.0
    common = set(r) & set(p)
    if not common:
        return 0.0
    prec = len(common) / len(p)
    rec  = len(common) / len(r)
    return 2 * prec * rec / (prec + rec)

df['token_f1'] = [token_f1(r, p) for r, p in zip(references, predictions)]

# --- ROUGE-1/2/L ---
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=False)
r1, r2, rL = [], [], []
for ref, pred in zip(references, predictions):
    s = scorer.score(ref, pred)
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rL.append(s['rougeL'].fmeasure)
df['rouge1_f1'] = r1
df['rouge2_f1'] = r2
df['rougeL_f1'] = rL

# --- MiniLM Semantic Similarity ---
print("MiniLM hesaplanıyor...")
sim_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
emb_pred = sim_model.encode(predictions, batch_size=64, convert_to_tensor=True)
emb_ref  = sim_model.encode(references,  batch_size=64, convert_to_tensor=True)
sims = util.cos_sim(emb_pred, emb_ref).diagonal().cpu().numpy()
df['minilm_sim'] = sims

# --- BERTScore (XLM-RoBERTa-base) ---
print("BERTScore hesaplanıyor...")
P, R, F1 = bert_score(predictions, references,
                      lang='tr', model_type='xlm-roberta-base', verbose=True)
df['bertscore_f1'] = F1.numpy()

# --- Generated Length ---
df['gen_tokens'] = [len(str(p).split()) for p in predictions]

df.to_csv("qwen3_finetuned_metrics.csv", index=False)

# --- Özet Karşılaştırma Tablosu ---
baseline = {
    'Exact Match':  0.0000,
    'Token F1':     0.0653,
    'ROUGE-1 F1':   0.1109,
    'ROUGE-2 F1':   0.0140,
    'ROUGE-L F1':   0.0757,
    'MiniLM Sim.':  0.3798,
    'BERTScore F1': 0.8191,
    'Avg Tokens':   36.4,
}
results = {
    'Exact Match':  df['exact_match'].mean(),
    'Token F1':     df['token_f1'].mean(),
    'ROUGE-1 F1':   df['rouge1_f1'].mean(),
    'ROUGE-2 F1':   df['rouge2_f1'].mean(),
    'ROUGE-L F1':   df['rougeL_f1'].mean(),
    'MiniLM Sim.':  df['minilm_sim'].mean(),
    'BERTScore F1': df['bertscore_f1'].mean(),
    'Avg Tokens':   df['gen_tokens'].mean(),
}

print("\n" + "="*58)
print(f"{'METRİK':<20} {'BASELINE':>10} {'POST-SFT':>10} {'DELTA':>10}")
print("="*58)
for k in baseline:
    b, f = baseline[k], results[k]
    d = f - b
    print(f"{k:<20} {b:>10.4f} {f:>10.4f} {'↑' if d>0 else '↓'}{abs(d):>8.4f}")

# --- Alan Bazında Token F1 ---
print("\n=== ALAN BAZINDA TOKEN F1 ===")
baseline_alan = {
    'beyin-ve-sinir-cerrahisi':       0.0570,
    'uroloji':                        0.0590,
    'ortopedi-ve-travmatoloji':       0.0701,
    'genel-cerrahi':                  0.0694,
    'kulak-burun-bogaz-hastaliklari': 0.0632,
    'fiziksel-tip-ve-rehabilitasyon': 0.0852,
    'kardiyoloji':                    0.0846,
    'kalp-damar-cerrahisi':           0.0573,
    'dahiliye-ve-ic-hastaliklari':    0.0640,
}
alan = df.groupby('doctor_speciality')['token_f1'].mean().round(4)
print(f"{'Alan':<40} {'Baseline':>10} {'Post-SFT':>10} {'Delta':>8}")
print("-"*72)
for alan_adi, ft_val in alan.items():
    base_val = baseline_alan.get(alan_adi, 0)
    delta = ft_val - base_val
    print(f"{alan_adi:<40} {base_val:>10.4f} {ft_val:>10.4f} {'↑' if delta>0 else '↓'}{abs(delta):>6.4f}")